***

*Course:* [Math 535](https://people.math.wisc.edu/~roch/mmids/) - Mathematical Methods in Data Science (MMiDS)  
*Chapter:* 6-Optimization theory and algorithms   
*Author:* [Sebastien Roch](https://people.math.wisc.edu/~roch/), Department of Mathematics, University of Wisconsin-Madison  
*Updated:* Jan 6, 2024   
*Copyright:* &copy; 2024 Sebastien Roch

***

In [ ]:
# IF RUNNING ON GOOGLE COLAB, UNCOMMENT THE FOLLOWING CODE CELL
# When prompted, upload: 
#     * mmids.py
#     * advertising.csv 
# from your local file system
# Files at: https://github.com/MMiDS-textbook/MMiDS-textbook.github.io/tree/main/utils
# Alternative instructions: https://colab.research.google.com/notebooks/io.ipynb

In [ ]:
#from google.colab import files
#
#uploaded = files.upload()
#
#for fn in uploaded.keys():
#    print('User uploaded file "{name}" with length {length} bytes'.format(
#      name=fn, length=len(uploaded[fn])))

In [ ]:
# FOR BOOK ONLY
import warnings
warnings.filterwarnings('ignore')
import os, sys
sys.path.insert(0, os.path.abspath('../../utils')) # use directory to mmids.py
#plt.rcParams['figure.figsize'] = [4.00, 2.25] # for high-def figs
#plt.rcParams['figure.dpi'] = 200 # for high-def figs

In [ ]:
# PYTHON 3
import numpy as np
from numpy import linalg as LA
from numpy.random import default_rng
rng = default_rng(535)
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
import tensorflow as tf
from tensorflow import keras
import mmids

In [ ]:
# FOR TeX ONLY
#plt.rcParams['figure.figsize'] = [4.00, 2.25] # for high-def figs
#plt.rcParams['figure.dpi'] = 200 # for high-def figs

$\newcommand{\bSigma}{\boldsymbol{\Sigma}}$
$\newcommand{\bmu}{\boldsymbol{\mu}}$

## Optimality conditions and convexity

In this section, we derive optimality conditions for unconstrained continuous optimization problems.

We will be interested in unconstrained optimization of the form:

$$
\min_{\mathbf{x} \in \mathbb{R}^d} f(\mathbf{x})
$$

where $f : \mathbb{R}^d \to \mathbb{R}$. In this subsection, we define several notions of solution and derive characterizations.

Recall the definition of a global minimizer.

**DEFINITION** **(Global minimizer)** Let $f : \mathbb{R}^d \to \mathbb{R}$. The point $\mathbf{x}^* \in \mathbb{R}^d$ is a global minimizer of $f$ over $\mathbb{R}^d$ if 

$$
f(\mathbf{x}) 
\geq f(\mathbf{x}^*), \quad \forall \mathbf{x} \in \mathbb{R}^d. 
$$

$\natural$

We have observed before that, in general, finding a global minimizer and certifying that one has been found can be difficult unless some special structure is present. Therefore weaker notions of solution are needed. Recall the notion of a local minimizer.

**DEFINITION** **(Local minimizer)** Let $f : \mathbb{R}^d \to \mathbb{R}$. The point $\mathbf{x}^* \in \mathbb{R}^d$ is a local minimizer of $f$ over $\mathbb{R}^d$ if there is $\delta > 0$ such that 

$$
f(\mathbf{x}) 
\geq f(\mathbf{x}^*), \quad \forall \mathbf{x} \in B_{\delta}(\mathbf{x}^*) \setminus \{\mathbf{x}^*\}. 
$$

If the inequality is strict, we say that $\mathbf{x}^*$ is a strict local minimizer. $\natural$

In words, $\mathbf{x}^*$ is a local minimizer if there is open ball around $\mathbf{x}^*$ where it attains the minimum value. The difference between global and local minimizers is illustrated in the next figure.

![Local and global optima](https://upload.wikimedia.org/wikipedia/commons/thumb/6/68/Extrema_example_original.svg/600px-Extrema_example_original.svg.png)

**Figure:** Local and global optima. ([Source](https://commons.wikimedia.org/wiki/File:Extrema_example_original.svg))

<!--TEX
![Local and global optima. ([Source](https://commons.wikimedia.org/wiki/File:Extrema_example_original.svg))](./figs/Extrema_example_original.svg.png)
-->

### First-order conditions

Local minimizers can be characterized in terms of the gradient. We first define the concept of directional derivative.

**Directional derivative** Partial derivatives measure the rate of change of a function along the axes. More generally:

**DEFINITION** **(Directional Derivative)** Let $f : D \to \mathbb{R}$ where $D \subseteq \mathbb{R}^d$, let $\mathbf{x}_0 \in D$ be an interior point of $D$ and let $\mathbf{v} \in \mathbb{R}^d$ be a nonzero vector. The directional derivative of $f$ at $\mathbf{x}_0$ in the direction $\mathbf{v}$ is 

$$
\frac{\partial f (\mathbf{x}_0)}{\partial \mathbf{v}} 
= \lim_{h \to 0} \frac{f(\mathbf{x}_0 + h \mathbf{v}) - f(\mathbf{x}_0)}{h}
$$

provided the limit exists. $\natural$

Note that taking $\mathbf{v} = \mathbf{e}_i$ recovers the $i$-th partial derivative

$$
\frac{\partial f (\mathbf{x}_0)}{\partial \mathbf{e}_i} 
= \lim_{h \to 0} \frac{f(\mathbf{x}_0 + h \mathbf{e}_i) - f(\mathbf{x}_0)}{h}
= \frac{\partial f (\mathbf{x}_0)}{\partial x_i}.
$$

Conversely, a general directional derivative can be expressed in terms of the partial derivatives.

**THEOREM** **(Directional Derivative from Gradient)** Let $f : D \to \mathbb{R}$ where $D \subseteq \mathbb{R}^d$, let $\mathbf{x}_0 \in D$ be an interior point of $D$ and let $\mathbf{v} \in \mathbb{R}^d$ be a vector. Assume that $f$ is continuously differentiable at $\mathbf{x}_0$. Then the directional derivative of $f$ at $\mathbf{x}_0$ in the direction $\mathbf{v}$ is given by

$$
\frac{\partial f (\mathbf{x}_0)}{\partial \mathbf{v}} 
= \nabla f(\mathbf{x}_0)^T \mathbf{v}.
$$

$\sharp$

*Proof idea:* To bring out the partial derivatives, we re-write the directional derivative as the derivative of a composition of $f$ with an affine function. We then use the *Chain Rule*.

*Proof:* Consider the composition $\beta(h) = f(\boldsymbol{\alpha}(h))$ where $\boldsymbol{\alpha}(h) = \mathbf{x}_0 + h \mathbf{v}$. Observe that $\boldsymbol{\alpha}(0)= \mathbf{x}_0$ and $\beta(0)= f(\mathbf{x}_0)$. Then, by definition of the derivative,

$$
\frac{\mathrm{d} \beta(0)}{\mathrm{d} h}
= \lim_{h \to 0} \frac{\beta(h) - \beta(0)}{h}
= \lim_{h \to 0} \frac{f(\mathbf{x}_0 + h \mathbf{v}) - f(\mathbf{x}_0)}{h}
= \frac{\partial f (\mathbf{x}_0)}{\partial \mathbf{v}}.
$$

Applying the *Chain Rule* and the *Parametric Line Example* from the previous section, we arrive at 

$$
\frac{\mathrm{d} \beta(0)}{\mathrm{d} h}
= \nabla f(\boldsymbol{\alpha}(0))^T 
\boldsymbol{\alpha}'(0)
= \nabla f(\mathbf{x}_0)^T 
\mathbf{v}.
$$

$\square$

Here is an example:

![Directional Derivative](https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Directional_derivative_contour_plot.svg/485px-Directional_derivative_contour_plot.svg.png)

**Figure:** Directional derivative ([Source](https://commons.wikimedia.org/wiki/File:Directional_derivative_contour_plot.svg))

<!--TEX

![Directional derivative ([Source](https://commons.wikimedia.org/wiki/File:Directional_derivative_contour_plot.svg))](./figs/Directional_derivative_contour_plot.svg.png)

-->

**Descent direction** Earlier in the book, we proved a key insight about the derivative of a single-variable function $f$ at a point $x_0$: it tells us where to find smaller values.

**LEMMA** **(Descent Direction)** Let $f : D \to \mathbb{R}$ with $D \subseteq \mathbb{R}$ and let $x_0 \in D$ be an interior point of $D$ where $f'(x_0)$ exists. If $f'(x_0) > 0$, then there is an open ball $B_\delta(x_0) \subseteq D$ around $x_0$
such that for each $x$ in $B_\delta(x_0)$:

(a) $f(x) > f(x_0)$ if $x > x_0$,

(b) $f(x) < f(x_0)$ if $x < x_0$.

If instead $f'(x_0) < 0$, the opposite holds. $\flat$

We generalize the *Descent Direction Lemma* to the multivariable case. We first need to define what a descent direction is. 

**DEFINITION** **(Descent Direction)** Let $f : \mathbb{R}^d \to \mathbb{R}$. A vector $\mathbf{v}$ is a descent direction for $f$ at $\mathbf{x}_0$ if there is $\alpha^* > 0$ such that

$$
f(\mathbf{x}_0 + \alpha \mathbf{v})
< f(\mathbf{x}_0), \quad \forall \alpha \in (0,\alpha^*).
$$

$\natural$

In the continuously differentiable case, the directional derivative gives a criterion for descent directions.

**LEMMA** **(Descent Direction and Directional Derivative)** Let $f : \mathbb{R}^d \to \mathbb{R}$ be continuously differentiable at $\mathbf{x}_0$. A vector $\mathbf{v}$ is a descent direction for $f$ at $\mathbf{x}_0$ if

$$
\frac{\partial f (\mathbf{x}_0)}{\partial \mathbf{v}} 
= \nabla f(\mathbf{x}_0)^T \mathbf{v}
< 0
$$

that is, if the directional derivative of $f$ at $\mathbf{x}_0$ in the direction $\mathbf{v}$ is negative. $\flat$

*Proof idea:* In anticipation of the proof of the second-order condition, we use the *Mean Value Theorem* to show that $f$ takes smaller values in direction $\mathbf{v}$. For a simpler proof based on the definition of the directional derivative, see *Exercise 4.73*.

*Proof:* Suppose there is $\mathbf{v} \in \mathbb{R}^d$ such that $\nabla f(\mathbf{x}_0)^T \mathbf{v} = -\eta < 0$. For $\alpha > 0$, the *Mean Value Theorem* implies that there is $\xi_\alpha \in (0,1)$ such that

$$
f(\mathbf{x}_0 + \alpha \mathbf{v})
= f(\mathbf{x}_0) + \nabla f(\mathbf{x}_0 + \xi_\alpha \alpha \mathbf{v})^T(\alpha \mathbf{v})
= f(\mathbf{x}_0) + \alpha  \nabla f(\mathbf{x}_0 + \xi_\alpha \alpha \mathbf{v})^T \mathbf{v}.
$$

We want to show that the second term on the right-hand side is negative. We cannot immediately apply our condition on $\mathbf{v}$ as the gradient in the previous equation is taken at $\mathbf{x}_0 + \xi_\alpha \alpha \mathbf{v}$, not $\mathbf{x}_0$.

The gradient is continuous (in the sense that all its components are continuous). In particular, the function $\nabla f(\mathbf{x})^T \mathbf{v}$ is continuous as a linear combination of continuous functions. By the definition of continuity, for any $\epsilon > 0$ - say $\epsilon = \eta/2$ - there is $\delta > 0$ small enough such that all $\mathbf{x} \in B_\delta(\mathbf{x}_0)$ satisfy

\begin{align*}
\left|\nabla f(\mathbf{x})^T \mathbf{v}
- \nabla f(\mathbf{x}_0)^T \mathbf{v}\,\right|
&< \epsilon = \eta/2.
\end{align*}


Take $\alpha^* > 0$ small enough that $\mathbf{x}_0 + \alpha^* \mathbf{v} \in B_\delta(\mathbf{x}_0)$. Then, for all $\alpha \in (0,\alpha^*)$, whatever $\xi_\alpha \in (0,1)$ is, it holds that $\mathbf{x}_0 + \xi_\alpha \alpha \mathbf{v} \in B_\delta(\mathbf{x}_0)$. Hence,

\begin{align*}
\nabla f(\mathbf{x}_0 + \xi_\alpha \alpha \mathbf{v})^T \mathbf{v}
&= \nabla f(\mathbf{x}_0)^T \mathbf{v} + (\nabla f(\mathbf{x}_0 + \xi_\alpha \alpha \mathbf{v})^T \mathbf{v} - \nabla f(\mathbf{x}_0)^T \mathbf{v})\\
&\leq \nabla f(\mathbf{x}_0)^T \mathbf{v} + 
\left|\nabla f(\mathbf{x}_0 + \xi_\alpha \alpha \mathbf{v})^T \mathbf{v} - \nabla f(\mathbf{x}_0)^T \mathbf{v}\,\right|\\
&<  -\eta + \eta/2\\
&= - \eta/2 < 0.
\end{align*}

by definition of $\eta$. That implies

$$
f(\mathbf{x}_0 + \alpha \mathbf{v})
< f(\mathbf{x}_0) - \alpha \eta/2 < f(\mathbf{x}_0),
\quad \forall \alpha \in (0,\alpha^*)
$$

and proves the claim. $\square$

**LEMMA** **(Descent Direction: Multivariable Version)** Let $f : \mathbb{R}^d \to \mathbb{R}$ be continuously differentiable at $\mathbf{x}_0$ and assume that $\nabla f(\mathbf{x}_0) \neq 0$. Then $f$ has a descent direction at $\mathbf{x}_0$. $\flat$

*Proof:* Take $\mathbf{v} = - \nabla f(\mathbf{x}_0)$. Then $\nabla f(\mathbf{x}_0)^T \mathbf{v} = - \|\nabla f(\mathbf{x}_0)\|^2 < 0$ since $\nabla f(\mathbf{x}_0) \neq \mathbf{0}$. $\square$

This leads to the following fundamental result.

**THEOREM** **(First-Order Necessary Condition)** Let $f : \mathbb{R}^d \to \mathbb{R}$ be continuously differentiable on $\mathbb{R}^d$. If $\mathbf{x}_0$ is a local minimizer, then $\nabla f(\mathbf{x}_0) = \mathbf{0}$. $\sharp$

*Proof idea:* In a descent direction, $f$ decreases hence there cannot be one at a local minimizer. 

*Proof:* We argue by contradiction. Suppose that $\nabla f(\mathbf{x}_0) \neq 0$. By the *Descent Direction Lemma*, there is a descent direction $\mathbf{v} \in \mathbb{R}^d$ at $\mathbf{x}_0$. That implies

$$
f(\mathbf{x}_0 + \alpha \mathbf{v})
< f(\mathbf{x}_0), \quad \forall \alpha \in (0, \alpha^*) 
$$

for some $\alpha^* > 0$. So every open ball around $\mathbf{x}_0$ has a point achieving a smaller value than $f(\mathbf{x}_0)$. Thus $\mathbf{x}_0$ is not a local minimizer, a contradiction. So it must be that $\nabla f(\mathbf{x}_0) = \mathbf{0}$. $\square$

A point satisfying the first-order necessary condition is called a stationary point.

**DEFINITION** **(Stationary Point)** Let $f : D \to \mathbb{R}$ be continuously differentiable on an open set $D \subseteq \mathbb{R}^d$. If $\nabla f(\mathbf{x}_0) = \mathbf{0}$, we say that $\mathbf{x}_0 \in D$ is a stationary point of $f$. $\natural$

**EXAMPLE:** **(Rayleight Quotient)** Here we revisit the Rayleigh quotient. Let $A \in \mathbb{R}^{d \times d}$ be a symmetric matrix. Recall that the Rayleigh quotient is defined as

$$
\mathcal{R}_A(\mathbf{u})
= \frac{\langle \mathbf{u}, A \mathbf{u} \rangle}{\langle \mathbf{u}, \mathbf{u} \rangle}
$$

which is defined for any $\mathbf{u} = (u_1,\ldots,u_d) \neq \mathbf{0}$ in $\mathbb{R}^{d}$. As a function from $\mathbb{R}^{d}$ to $\mathbb{R}\setminus \{\mathbf{0}\}$, $\mathcal{R}_A(\mathbf{u})$ is continuously differentiable. We find its stationary points. 

We use the [quotient rule](https://en.wikipedia.org/wiki/Quotient_rule) and our previous results on the gradient of quadratic functions. Specifically, note that (using that $A$ is symmetric)

\begin{align*}
\frac{\partial}{\partial u_i} \mathcal{R}_A(\mathbf{u})
&= \frac{\left(\frac{\partial}{\partial u_i} \langle \mathbf{u}, A \mathbf{u} \rangle\right) \langle \mathbf{u}, \mathbf{u} \rangle
- \langle \mathbf{u}, A \mathbf{u} \rangle \left( \frac{\partial}{\partial u_i} \langle \mathbf{u}, \mathbf{u} \rangle\right)}{\langle \mathbf{u}, \mathbf{u} \rangle^2}\\
&= \frac{2\left(\frac{\partial}{\partial u_i} \frac{1}{2}\mathbf{u}^T A \mathbf{u}\right) \|\mathbf{u}\|^2
- \mathbf{u}^T A \mathbf{u} \left( \frac{\partial}{\partial u_i} \sum_{j=1}^d u_j^2\right)}{\|\mathbf{u}\|^4}\\
&= \frac{2\left(A \mathbf{u}\right)_i \|\mathbf{u}\|^2
- \mathbf{u}^T A \mathbf{u} \left( 2 u_i \right)}{\|\mathbf{u}\|^4}\\
&= \frac{2}{\|\mathbf{u}\|^2}\left\{\left(A \mathbf{u}\right)_i
- \mathcal{R}_A(\mathbf{u}) u_i \right\}.
\end{align*}


In vector form this is

$$
\nabla \mathcal{R}_A(\mathbf{u})
= \frac{2}{\|\mathbf{u}\|^2} \left\{A \mathbf{u}
- \mathcal{R}_A(\mathbf{u}) \,\mathbf{u} \right\}.
$$

The stationary points satisfy $\nabla \mathcal{R}_A(\mathbf{u})
= \mathbf{0}$, or after getting rid of the denominator and rearranging, 

$$
A \mathbf{u}
= \mathcal{R}_A(\mathbf{u}) \,\mathbf{u}.
$$


We already know the solutions to this system: the eigenvectors of $A$. Indeed, if $\mathbf{q}_i$ is a unit eigenvector of $A$ with eigenvalue $\lambda_i$, then we have seen that $\mathcal{R}_A(\mathbf{q}_i) = \lambda_i$ and we get

$$
A \mathbf{q}_i
= \mathcal{R}_A(\mathbf{q}_i) \,\mathbf{q}_i.
$$

We know from *Courant-Fischer* that the eigenvectors of $A$ are not in general local minimizers of its Rayleigh quotient - in fact one of them is a global maximizer! $\lhd$

### Second-order conditions

Local minimizers can also be characterized in terms of the Hessian.

We will make use of *Taylor's Theorem*, a generalization of the *Mean Value Theorem* that provides polynomial approximations to a function around a point. We restrict ourselves to the case of a linear approximation with second-order error term, which will suffice for our purposes. 

**Taylor's theorem** We begin by reviewing the single-variable case, which we will use to prove the general verison.

**THEOREM** **(Taylor)** Let $f: D \to \mathbb{R}$ where $D \subseteq \mathbb{R}$. Suppose $f$ has a continuous derivative on $[a,b]$ and that its second derivative exists on $(a,b)$. Then for any $x \in [a, b]$

$$
f(x)
= f(a) + (x-a) f'(a) + \frac{1}{2} (x-a)^2 f''(\xi)
$$

for some $a < \xi < x$. $\sharp$

The third term on the right-hand side of *Taylor's Theorem* is called the Lagrange remainder. It can be seen as an error term between $f(x)$ and the linear approximation $f(a) + (x-a) f'(a)$. There are [other forms](https://en.wikipedia.org/wiki/Taylor%27s_theorem#Explicit_formulas_for_the_remainder) for the remainder. The form we stated here is useful when one has a bound on the second derivative. Here is an example.

**EXAMPLE:** Consider $f(x) = e^x$. Then $f'(x) = f''(x) = e^x$. Suppose we are interested in approximating $f$ in the interval $[0,1]$. We take $a=0$ and $b=1$ in *Taylor's Theorem*. The linear term is 

$$
f(a) + (x-a) f'(a) = 1 + x e^0 = 1 + x.
$$

Then for any $x \in [0,1]$

$$
f(x) = 1 + x + \frac{1}{2}x^2 e^{\xi_x}
$$

where $\xi_x \in (0,1)$ depends on $x$. We get a uniform bound on the error over $[0,1]$ by replacing $\xi_x$ with its worst possible value over $[0,1]$ 

$$
|f(x) - (1+x)| \leq \frac{1}{2}x^2 e^{\xi_x} \leq \frac{e}{2} x^2.
$$

In [ ]:
x = np.linspace(0,1,100)
y = np.exp(x)
taylor = 1 + x
err = (np.exp(1)/2) * x**2

In [ ]:
plt.plot(x,y,label='f')
plt.plot(x,taylor,label='taylor')
plt.legend()
plt.show()

If we plot the upper and lower bounds, we see that $f$ indeed falls within them.

In [ ]:
plt.plot(x,y,label='f')
plt.plot(x,taylor,label='taylor')
plt.plot(x,taylor-err,linestyle=':',color='green',label='lower')
plt.plot(x,taylor+err,linestyle='--',color='green',label='upper')
plt.legend()
plt.show()

$\lhd$

In the case of several variables, we again restrict ourselves to the second order. For the more general version, see e.g. [Wikipedia](https://en.wikipedia.org/wiki/Taylor's_theorem#Taylor's_theorem_for_multivariate_functions). 

**THEOREM** **(Taylor)** Let $f : D \to \mathbb{R}$ where $D \subseteq \mathbb{R}^d$. Let $\mathbf{x}_0 \in D$ and $\delta > 0$ be such that $B_\delta(\mathbf{x}_0) \subseteq D$. If $f$ is twice continuously differentiable on $B_\delta(\mathbf{x}_0)$, then for any $\mathbf{x} \in B_\delta(\mathbf{x}_0)$

$$
f(\mathbf{x})
= f(\mathbf{x}_0) + \nabla f(\mathbf{x}_0)^T (\mathbf{x} - \mathbf{x}_0) 
+ \frac{1}{2} (\mathbf{x} - \mathbf{x}_0)^T 
\,\mathbf{H}_f(\mathbf{x}_0 + \xi (\mathbf{x} - \mathbf{x}_0))
\,(\mathbf{x} - \mathbf{x}_0),
$$

for some $\xi \in (0,1)$. $\sharp$

As in the single-variable case, we think of $f(\mathbf{x}_0) + \nabla f(\mathbf{x}_0)^T (\mathbf{x} - \mathbf{x}_0)$ for fixed $\mathbf{x}_0$ as a linear - or more accurately affine - approximation to $f$ at $\mathbf{x}_0$. The third term on the right-hand side above quantifies the error of this approximation. 

*Proof idea:* We apply the single-variable result to $\phi(t) = f(\boldsymbol{\alpha}(t))$. We use the *Chain Rule* to compute the needed derivatives.

*Proof:* Let $\mathbf{p} = \mathbf{x} - \mathbf{x}_0$ and $\phi(t) = f(\boldsymbol{\alpha}(t))$ where $\boldsymbol{\alpha}(t) = \mathbf{x}_0 + t \mathbf{p}$. Observe that $\phi(0) = f(\mathbf{x}_0)$ and $\phi(1) = f(\mathbf{x})$. As observed in the proof of the *Mean Value Theorem*, $\phi'(t) = \nabla f(\boldsymbol{\alpha}(t))^T \mathbf{p}$. By the *Chain Rule* and our previous *Parametric Line Example*,

\begin{align*}
\phi''(t)
&= \frac{\mathrm{d}}{\mathrm{d} t}
\left[\sum_{i=1}^d \frac{\partial f(\boldsymbol{\alpha}(t))}{\partial x_i} p_i \right]\\
&= 
\sum_{i=1}^d \left(\nabla \frac{\partial f(\boldsymbol{\alpha}(t))}{\partial x_i}\right)^T \boldsymbol{\alpha}'(t) \,p_i \\
&= \sum_{i=1}^d \sum_{j=1}^d \frac{\partial^2 f(\boldsymbol{\alpha}(t))}{\partial x_j \partial x_i}  p_j p_i\\
&= \mathbf{p}^T 
\,\mathbf{H}_f(\mathbf{x}_0 + t \mathbf{p})
\,\mathbf{p}.
\end{align*}

In particular, $\phi$ has continuous first and second derivatives on $[0,1]$.

By *Taylor's Theorem* in the single-variable case

$$
\phi(t)
= \phi(0) + t \phi'(0) + \frac{1}{2} t^2 \phi''(\xi)
$$

for some $\xi \in (0,t)$. Plugging in the expressions for $\phi(0)$, $\phi'(0)$ and $\phi''(\xi)$ and taking $t=1$ gives the claim. $\square$

**EXAMPLE:** Consider the function $f(x_1, x_2) = x_1 x_2 + x_1^2 + e^{x_1} \cos x_2$. We apply *Taylor's Theorem* with $\mathbf{x}_0 = (0, 0)$ and $\mathbf{x} = (x_1, x_2)$. The gradient is

$$
\nabla f(x_1, x_2) = (x_2 + 2 x_1 + e^{x_1} \cos x_2, x_1 - e^{x_1} \sin x_2 )^T
$$

and the Hessian is

$$
\mathbf{H}_f(x_1, x_2)
= \begin{pmatrix}
2 + e^{x_1} \cos x_2 & 1 - e^{x_1} \sin x_2\\
1 - e^{x_1} \sin x_2 & - e^{x_1} \cos x_2 
\end{pmatrix}.
$$

So $f(0,0) = 1$ and $\nabla f(0,0) = (1, 0)^T$. Thus, 
by *Taylor's Theorem*, there is $\xi \in (0,1)$ such that

$$
f(x_1, x_2)
= 1 + x_1 
+ \frac{1}{2}[2 x_1^2 + 2 x_1 x_2 
+ (x_1^2 - x_2^2) \,e^{\xi x_1} \cos(\xi x_2)
- 2 x_1 x_2 e^{\xi x_1} \sin(\xi x_2)].
$$

$\lhd$

**Weyl's inequality** We will also need a bound on the eigenvalues of symmetric matrices. The proof is in an optional section below.

For a symmetric matrix $C \in  \mathbb{R}^{d \times d}$, we let $\lambda_j(C)$, $j=1, \ldots, d$, be the eigenvalues of $C$ in non-increasing order with corresponding orthonormal eigenvectors $\mathbf{v}_j$, $j=1, \ldots, d$. The following lemma is one version of what is known as *Weyl's Inequality*.

**LEMMA** **(Weyl)** Let $A \in \mathbb{R}^{d \times d}$ and $B \in \mathbb{R}^{d \times d}$ be symmetric matrices. Then, for all $j=1, \ldots, d$, 

$$
\max_{j \in [d]} \left|\lambda_j(B) - \lambda_j(A)\right|
\leq \|B- A\|_2
$$

where $\|C\|_2$ is the induced $2$-norm of $C$. $\flat$

**Necessary condition** When $f$ is twice continuously differentiable, we get a necessary condition based on the Hessian.

**THEOREM** **(Second-Order Necessary Condition)** Let $f : \mathbb{R}^d \to \mathbb{R}$ be twice continuously differentiable on $\mathbb{R}^d$. If $\mathbf{x}_0$ is a local minimizer, then $\nabla f(\mathbf{x}_0) = \mathbf{0}$ and $\mathbf{H}_f(\mathbf{x}_0)$ is positive semidefinite. $\sharp$

*Proof idea:* By *Taylor* and the *First-Order Necessary Condition*,

\begin{align*}
f(\mathbf{x}_0 + \alpha \mathbf{v})
&= f(\mathbf{x}_0) 
+ \nabla f(\mathbf{x}_0)^T(\alpha \mathbf{v})
+ \frac{1}{2}(\alpha \mathbf{v})^T \mathbf{H}_f(\mathbf{x}_0 + \xi_\alpha \alpha \mathbf{v}) (\alpha \mathbf{v})\\
&= f(\mathbf{x}_0) 
+ \frac{1}{2}\alpha^2 \mathbf{v}^T \mathbf{H}_f(\mathbf{x}_0 + \xi_\alpha \alpha \mathbf{v})\,\mathbf{v}.
\end{align*}

If $\mathbf{H}_f$ is positive semidefinite in a neighborhood around $\mathbf{x}_0$, then the second term on the right-hand side is nonnegative, which is necessary for $\mathbf{x}_0$ to be a local minimizer. Formally we argue  by contradiction. The proof goes along the same lines as the argument leading to the *First-Order Necessary Condition*.

*Proof:* We argue by contradiction. Suppose that $\mathbf{H}_f(\mathbf{x}_0)$ is not positive semidefinite. From the *Symmetry of the Hessian Theorem* and the *Spectral Theorem*, $\mathbf{H}_f(\mathbf{x}_0)$ has a spectral decomposition. From the *Characterization of Positive Semidefiniteness*, however, it follows that $\mathbf{H}_f(\mathbf{x}_0)$ must have at least one negative eigenvalue $- \eta < 0$. Let $\mathbf{v}$ be a corresponding unit eigenvector.

We have that $\langle \mathbf{v}, \mathbf{H}_f(\mathbf{x}_0) \,\mathbf{v} \rangle = - \eta < 0$. For $\alpha > 0$, *Taylor's Theorem* implies that there is $\xi_\alpha \in (0,1)$ such that

\begin{align*}
f(\mathbf{x}_0 + \alpha \mathbf{v})
&= f(\mathbf{x}_0) 
+ \nabla f(\mathbf{x}_0)^T(\alpha \mathbf{v})
+ \frac{1}{2} (\alpha \mathbf{v})^T \mathbf{H}_f(\mathbf{x}_0 + \xi_\alpha \alpha \mathbf{v}) (\alpha \mathbf{v})\\
&= f(\mathbf{x}_0) 
+ \frac{1}{2} \alpha^2 \mathbf{v}^T \mathbf{H}_f(\mathbf{x}_0 + \xi_\alpha \alpha \mathbf{v})\,\mathbf{v}
\end{align*}

where we used $\nabla f(\mathbf{x}_0) = \mathbf{0}$ by the *First-Order Necessary Condition*. We want to show that the second term on the right-hand side is negative.

The Hessian is continuous (in the sense that all its entries are continuous functions of $\mathbf{x}$). In particular, the function $ \mathbf{v}^T \mathbf{H}_f(\mathbf{x}) \,\mathbf{v}$ is continuous as a linear combination of continuous functions. So, by definition of continuity, for any $\epsilon > 0$ - say $\epsilon = \eta/2$ - there is $\delta > 0$ small enough that

\begin{align*}
\left| \mathbf{v}^T \mathbf{H}_f(\mathbf{x}) \,\mathbf{v}
- \mathbf{v}^T \mathbf{H}_f(\mathbf{x}_0) \,\mathbf{v} \right| 
&< \eta/2
\end{align*}

for all $\mathbf{x} \in B_\delta(\mathbf{x}_0)$.

Take $\alpha^* > 0$ small enough that $\mathbf{x}_0 + \alpha^* \mathbf{v} \in B_\delta(\mathbf{x}_0)$. Then, for all $\alpha \in (0,\alpha^*)$, whatever $\xi_\alpha \in (0,1)$ is, it holds that $\mathbf{x}_0 + \xi_\alpha \alpha \mathbf{v} \in B_\delta(\mathbf{x}_0)$. Hence,

\begin{align*}
\mathbf{v}^T \mathbf{H}_f(\mathbf{x}_0 + \xi_\alpha \alpha \mathbf{v}) \,\mathbf{v} 
&= \mathbf{v}^T \mathbf{H}_f(\mathbf{x}_0) \,\mathbf{v}  
+ (\mathbf{v}^T \mathbf{H}_f(\mathbf{x}_0 + \xi_\alpha \alpha \mathbf{v}) \,\mathbf{v}  
- \mathbf{v}^T \mathbf{H}_f(\mathbf{x}_0) \,\mathbf{v} )\\
&\leq \mathbf{v}^T \mathbf{H}_f(\mathbf{x}_0) \,\mathbf{v}  
+ |\mathbf{v}^T \mathbf{H}_f(\mathbf{x}_0 + \xi_\alpha \alpha \mathbf{v}) \,\mathbf{v}  
- \mathbf{v}^T \mathbf{H}_f(\mathbf{x}_0) \,\mathbf{v}|\\
&< -\eta 
+ \eta/2\\
&< - \eta/2 < 0.
\end{align*}

by definition of $\eta$. That implies

$$
f(\mathbf{x}_0 + \alpha \mathbf{v})
< f(\mathbf{x}_0) - \alpha^2 \eta/4 < f(\mathbf{x}_0). 
$$

Since this holds for all sufficiently small $\alpha$, every open ball around $\mathbf{x}_0$ has a point achieving a lower value than $f(\mathbf{x}_0)$. Thus $\mathbf{x}_0$ is not a local minimizer, a contradiction. So it must be that $\mathbf{H}_f(\mathbf{x}_0) \succeq \mathbf{0}$. $\square$

**Sufficient condition** The necessary condition above is not in general sufficient, as the following example shows.

**EXAMPLE:** Let $f(x) = x^3$. Then $f'(x) = 3 x^2$ and $f''(x) = 6 x$ so that $f'(0) = 0$ and $f''(0) \geq 0$. Hence $x=0$ is a stationary point. But $x=0$ is not a local minimizer. Indeed $f(0) = 0$ but, for any $\delta > 0$, $f(-\delta) < 0$.

In [ ]:
x = np.linspace(-2,2,100)
y = x**3

In [ ]:
plt.plot(x,y)
plt.ylim(-5,5)
plt.show()

$\lhd$

We give sufficient conditions for a point to be a local minimizer.

**THEOREM** **(Second-Order Sufficient Condition)** Let $f : \mathbb{R}^d \to \mathbb{R}$ be twice continuously differentiable on $\mathbb{R}^d$. If $\nabla f(\mathbf{x}_0) = \mathbf{0}$ and $\mathbf{H}_f(\mathbf{x}_0)$ is positive definite, then $\mathbf{x}_0$ is a strict local minimizer. $\sharp$

*Proof idea:* We use *Taylor's Theorem* again. This time we use the positive definiteness of the Hessian to bound the value of the function from below. We use *Weyl* to show that the eigenvalues of the Hessian are continuous, which implies that the Hessian remains positive definite in an open ball around $\mathbf{x}_0$.

*Proof:* By *Taylor's Theorem*, for all $\mathbf{v}$ there is $\xi_{\mathbf{v}} \in (0,1)$

\begin{align*}
f(\mathbf{x}_0 + \mathbf{v})
&= f(\mathbf{x}_0) 
+ \nabla f(\mathbf{x}_0)^T \mathbf{v}
+ \frac{1}{2} \mathbf{v}^T \,\mathbf{H}_f(\mathbf{x}_0 + \xi_{\mathbf{v}} \mathbf{v}) \,\mathbf{v}\\
&= f(\mathbf{x}_0) 
+ \frac{1}{2} \mathbf{v}^T \,\mathbf{H}_f(\mathbf{x}_0 + \xi_{\mathbf{v}} \mathbf{v}) \,\mathbf{v},
\end{align*}

where we used that $\nabla f(\mathbf{x}_0) = \mathbf{0}$. The second term on the last line is $0$ at $\mathbf{v} = \mathbf{0}$. Our goal is to show that is strictly positive in a neighborhood of $\mathbf{0}$. For this, we prove a uniform strictly positive lower bound on the eigenvalues of $\mathbf{H}_f$.

Formally, we claim that there is $\delta > 0$ such that $\mathbf{H}_f(\mathbf{x})$ is positive definite for all $\mathbf{x} \in B_{\delta}(\mathbf{x}_0)$. That is the case when $\mathbf{x} = \mathbf{x}_0$ and let $\mu := \lambda_d(\mathbf{H}_f(\mathbf{x}_0))> 0$ be the smallest eigenvalue of $\mathbf{H}_f(\mathbf{x}_0)$. We use *Weyl* to bound the eigenvalues of the Hessian from below around $\mathbf{x}_0$. For any vector $\mathbf{v} \in \mathbb{R}^d$, we have for all $j =1, \ldots, d$

$$
\lambda_j(\mathbf{H}_f(\mathbf{x}_0 + \mathbf{v}))
\geq \lambda_j(\mathbf{H}_f(\mathbf{x}_0)) - \|\mathbf{H}_f(\mathbf{x}_0 + \mathbf{v}) - \mathbf{H}_f(\mathbf{x}_0)\|_2
$$

where we used the notation introduced above.

We bound $\lambda_j(\mathbf{H}_f(\mathbf{x}_0))
\geq \lambda_d(\mathbf{H}_f(\mathbf{x}_0)) = \mu$ and

$$
\|\mathbf{H}_f(\mathbf{x}_0 + \mathbf{v}) - \mathbf{H}_f(\mathbf{x}_0)\|_2
\leq \|\mathbf{H}_f(\mathbf{x}_0 + \mathbf{v}) - \mathbf{H}_f(\mathbf{x}_0)\|_F.
$$

The Frobenius norm above is continuous in $\mathbf{v}$ as a composition of continuous functions. Moreover, we of course have at $\mathbf{v} = \mathbf{0}$ that this Frobenius norm is $0$. Hence, by definition of continuity, for any $\epsilon > 0$ - say $\epsilon := \mu/2$ - there is $\delta > 0$ such that $\mathbf{v} \in B_{\delta}(\mathbf{0})$ implies $\|\mathbf{H}_f(\mathbf{x}_0 + \mathbf{v}) - \mathbf{H}_f(\mathbf{x}_0)\|_F < \epsilon = \mu/2$. 

Plugging back above finally gives for any such $\mathbf{v}$ and all $j=1,\ldots,d$

$$
\lambda_j(\mathbf{H}_f(\mathbf{x}_0 + \mathbf{v})) > \mu/2 > 0.
$$

In particular, $\mathbf{H}_f(\mathbf{x}_0 + \mathbf{v})$ is positive definite and $\langle \mathbf{u}, \mathbf{H}_f(\mathbf{x}_0 + \mathbf{v}) \,\mathbf{u} \rangle > \frac{\mu}{2} \|\mathbf{u}\|^2$ for any $\mathbf{u}$ by *Courant-Fischer*.

Going back to our Taylor expansion, $\forall \mathbf{v} \in B_{\delta}(\mathbf{0}) \setminus \{\mathbf{0}\}$ there is $\xi_{\mathbf{v}} \in (0,1)$

\begin{align*}
f(\mathbf{x}_0 + \mathbf{v})
&= f(\mathbf{x}_0) 
+ \frac{1}{2} \mathbf{v}^T \,\mathbf{H}_f(\mathbf{x}_0 + \xi_{\mathbf{v}} \mathbf{v}) \,\mathbf{v}\\
&> f(\mathbf{x}_0) 
+ \frac{\mu}{4} \|\mathbf{v}\|^2\\
&> f(\mathbf{x}_0) 
\end{align*}

where we used that $\|\xi_{\mathbf{v}} \mathbf{v}\| = \xi_{\mathbf{v}} \|\mathbf{v}\| \leq \rho$. Therefore $\mathbf{x}_0$ is a strict local minimizer. $\square$

### Convexity

Our optimality conditions have only concerned local minimizers. Indeed, in the absence of global structure, local information such as gradients and Hessians can only inform us about the immediate neighborhood of points. Here we introduce convexity, a commonly encountered condition under which local minimizers become global minimizers.

**Convex sets** We start with convex sets.

**DEFINITION** **(Convex Set)** A set $D \subseteq \mathbb{R}^d$ is convex if for all $\mathbf{x}, \mathbf{y} \in D$ and all $\alpha \in (0,1)$

$$
(1-\alpha) \mathbf{x} + \alpha \mathbf{y} \in D.
$$

$\natural$

Note that, as $\alpha$ goes from $0$ to $1$,

$$
(1-\alpha) \mathbf{x} + \alpha \mathbf{y} = \mathbf{x} + \alpha (\mathbf{y} - \mathbf{x}),
$$

traces a line joining $\mathbf{x}$ and $\mathbf{y}$. In words, a set is convex if all segments between pairs of points in the set also lie in it.

The following is a convex set:

![A convex set ](https://upload.wikimedia.org/wikipedia/commons/thumb/6/6b/Convex_polygon_illustration1.svg/254px-Convex_polygon_illustration1.svg.png)

**Figure:** A convex set ([Source](https://commons.wikimedia.org/wiki/File:Convex_polygon_illustration1.svg))

<!--TEX

![A convex set ([Source](https://commons.wikimedia.org/wiki/File:Convex_polygon_illustration1.svg))](./figs/Convex_polygon_illustration1.svg.png)

-->

While the following is not:

![A set that is not convex](https://upload.wikimedia.org/wikipedia/commons/thumb/6/6c/Convex_polygon_illustration2.svg/254px-Convex_polygon_illustration2.svg.png)

**Figure:** A set that is not convex ([Source](https://commons.wikimedia.org/wiki/File:Convex_polygon_illustration2.svg))

<!--TEX

![A set that is not convex ([Source](https://commons.wikimedia.org/wiki/File:Convex_polygon_illustration2.svg))](./figs/1729px-Convex_polygon_illustration2.svg.png)

-->

**EXAMPLE:** An open ball in $\mathbb{R}^d$ is convex. Indeed, let $\delta > 0$ and $\mathbf{x}_0 \in \mathbb{R}^d$. For any $\mathbf{x}, \mathbf{y} \in B_{\delta}(\mathbf{x}_0)$ and any $\alpha \in [0,1]$, we have

\begin{align*}
\|[(1-\alpha) \mathbf{x} + \alpha \mathbf{y}] - \mathbf{x}_0\|_2
&= \|(1-\alpha) (\mathbf{x} - \mathbf{x}_0) + \alpha (\mathbf{y} - \mathbf{x}_0)\|_2\\
&\leq  \|(1-\alpha) (\mathbf{x} - \mathbf{x}_0)\|_2 + \|\alpha (\mathbf{y} - \mathbf{x}_0)\|_2\\
&= (1-\alpha) \|\mathbf{x} - \mathbf{x}_0\|_2 + \alpha \|\mathbf{y} - \mathbf{x}_0\|_2\\
&< (1-\alpha) \delta + \alpha \delta\\
&= \delta
\end{align*}

where we used the triangle inequality on the second line. Hence we have established that $(1-\alpha) \mathbf{x} + \alpha \mathbf{y} \in B_{\delta}(\mathbf{x}_0)$.

One remark. All we used in this argument is that the Euclidean norm is homogeneous and satisfies the triangle inequality. That is true of every norm. So we conclude that an open ball under any norm is convex. Also, the open nature of the set played no role. The same holds for closed balls in any norm. $\lhd$

**EXAMPLE:** Here is an important generalization. Think of the space of $n \times n$ symmetric matrices 

$$
\mathbf{S}^n 
= \left\{
X \in \mathbb{R}^{n \times n}\,:\, X = X^T
\right\},
$$

as a linear subspace of $\mathbb{R}^{n^2}$ (How?). The dimension of $\mathbf{S}^n$ is ${n \choose 2} + n$, the number of free parameters under the symmetry assumption. Consider now the set of all positive semidefinite matrices in $\mathbf{S}^n$

$$
\mathbf{S}_+^n 
= \left\{
X \in \mathbf{S}^n \,:\, X \succeq \mathbf{0}
\right\}.
$$


(Recall that $\mathbf{S}_+^n$ is not the same as the set of symmetric matrices with nonnegative elements.)

We claim that the set $\mathbf{S}_+^n$ is convex. Indeed let $X, Y \in \mathbf{S}_+^n$ and $\alpha \in [0,1]$. Then by postive semidefiniteness of $X$ and $Y$, for any $\mathbf{v} \in \mathbb{R}^n$

$$
\langle \mathbf{v}, [(1-\alpha) X + \alpha Y] \mathbf{v}\rangle
= (1-\alpha) \langle \mathbf{v}, X \mathbf{v}\rangle
+ \alpha \langle \mathbf{v}, Y \mathbf{v}\rangle
\geq 0.
$$

This shows that $(1-\alpha) X + \alpha Y \succeq \mathbf{0}$ and hence that $\mathbf{S}_+^n$ is convex. $\lhd$

A number of operations preserve convexity. In an abuse of notation, we think of a pair of vectors $(\mathbf{x}_1, \mathbf{x}_2) \in \mathbb{R}^d \times \mathbb{R}^{f}$ as a vector in $\mathbb{R}^{d+f}$. Put differently, $(\mathbf{x}_1, \mathbf{x}_2)$ is the vertical concatenation of column vectors $\mathbf{x}_1$ and $\mathbf{x}_2$. This is not to be confused with $\begin{pmatrix}\mathbf{x}_1 & \mathbf{x}_2\end{pmatrix}$ which is the $d \times 2$ matrix with columns $\mathbf{x}_1$ and $\mathbf{x}_2$. 

**LEMMA** **(Operations that Preserve Convexity)** Let $S_1, S_2 \subseteq \mathbb{R}^d$, $S_3 \subseteq \mathbb{R}^{f}$, and  $S_4 \subseteq \mathbb{R}^{d+f}$ be convex sets. Let $\beta \in \mathbb{R}$ and $\mathbf{b} \in \mathbb{R}^d$. The following sets are also convex:

(a) *Scaling:* $\beta S_1 = \{\beta \mathbf{x}\,:\, \mathbf{x} \in S_1\}$

(b) *Translation:* $S_1 + \mathbf{b} = \{\mathbf{x} + \mathbf{b}\,:\, \mathbf{x} \in S_1\}$

(c) *Sum:* $S_1 + S_2 = \{\mathbf{x}_1 + \mathbf{x}_2\,:\, \mathbf{x}_1 \in S_1 \text{ and } \mathbf{x}_2 \in S_2\}$

(d) *Cartesian product:* $S_1 \times S_3 = \{(\mathbf{x}_1, \mathbf{x}_2) \,:\, \mathbf{x}_1 \in S_1 \text{ and } \mathbf{x}_2 \in S_3\}$

(e) *Projection:* $T = \{\mathbf{x}_1\,:\, (\mathbf{x}_1, \mathbf{x}_2) \in S_4 \text{ for some }\mathbf{x}_2 \in \mathbb{R}^f\}$

(f) *Intersection:* $S_1 \cap S_2$

$\flat$

*Proof:* We only prove (f). The other statements are left as an exercise. Suppose $\mathbf{x}, \mathbf{y} \in S_1 \cap S_2$ and $\alpha \in [0,1]$. Then, by the convexity of $S_1$, $(1-\alpha) \mathbf{x} + \alpha \mathbf{y} \in S_1$ and, by the convexity of $S_2$, $(1-\alpha) \mathbf{x} + \alpha \mathbf{y} \in S_2$. Hence

$$
(1-\alpha) \mathbf{x} + \alpha \mathbf{y} \in S_1 \cap S_2.
$$

This property can be extended to an intersection of an arbitrary number of convex sets. $\square$

**Convex functions** Our main interest is in convex functions.

Here is the definition.

**DEFINITION** **(Convex Function)** A function $f : \mathbb{R}^d \to \mathbb{R}$ is convex if, for all $\mathbf{x}, \mathbf{y} \in \mathbb{R}^d$ and all $\alpha \in (0,1)$

$$
f((1-\alpha) \mathbf{x} + \alpha \mathbf{y})
\leq (1-\alpha) f(\mathbf{x}) + \alpha f(\mathbf{y}).
$$

More generally, a function $f : D \to \mathbb{R}$ over a convex domain $D \subseteq \mathbb{R}^d$ is convex if the definition above holds over all $\mathbf{x}, \mathbf{y} \in D$. A function is said to be strictly convex if a strict inequality holds. If $-f$ is convex (respectively, strictly convex), then $f$ is said to be concave (respectively, strictly concave). $\natural$

The definition above is sometimes referred to as the [secant line](https://en.wikipedia.org/wiki/Secant_line) definition (see the next figure). 

![A convex function ](https://upload.wikimedia.org/wikipedia/commons/thumb/c/c7/ConvexFunction.svg/1024px-ConvexFunction.svg.png)

**Figure:** A convex function ([Source](https://commons.wikimedia.org/wiki/File:ConvexFunction.svg))

<!--TEX

![A convex function ([Source](https://commons.wikimedia.org/wiki/File:ConvexFunction.svg))](https://upload.wikimedia.org/wikipedia/commons/thumb/c/c7/ConvexFunction.svg/1024px-ConvexFunction.svg.png)

-->

**LEMMA** **(Affine Functions are Convex)** Let $\mathbf{w} \in \mathbb{R}^d$ and $b \in \mathbb{R}$. The function $f(\mathbf{x}) = \mathbf{w}^T \mathbf{x} + b$ is convex. $\flat$

*Proof:* For any $\mathbf{x}, \mathbf{y} \in \mathbb{R}^d$ and $\alpha \in [0,1]$, 

$$
f((1-\alpha) \mathbf{x} + \alpha \mathbf{y})
= \mathbf{w}^T [(1-\alpha) \mathbf{x} + \alpha \mathbf{y}] + b
= (1-\alpha)[\mathbf{w}^T \mathbf{x} + b] + \alpha [\mathbf{w}^T \mathbf{y} + b]
$$

which proves the claim. $\square$

Here is a less straightforward example. A concrete application is given below. (See also *Exercise 4.47* for the supremum version of this claim - which does not require the constraint set to be convex!)

**LEMMA** **(Infimum over a Convex Set)** Let $f : \mathbb{R}^{d+f} \to \mathbb{R}$ be a convex function and let $C \subseteq \mathbb{R}^{f}$ be a convex set. The function

$$
g(\mathbf{x})
= \inf_{\mathbf{y} \in C} f(\mathbf{x},\mathbf{y}),
$$

is convex provided $g(\mathbf{x}) > -\infty$ for all $\mathbf{x} \in \mathbb{R}^d$. $\flat$

*Proof:* Let $\mathbf{x}_1, \mathbf{x}_2 \in \mathbb{R}^d$ and $\alpha \in [0,1]$. For $i=1,2$, by definition of $g$, for any $\epsilon > 0$ there is $\mathbf{y}_i \in C$ such that $f(\mathbf{x}_i, \mathbf{y}_i) \leq g(\mathbf{x}_i) + \epsilon$.  

By the convexity of $C$, $\alpha \mathbf{y}_1 + (1- \alpha)\mathbf{y}_2 \in C$. So because $g$ is an infimum over points $\mathbf{y}$ in $C$, we have

\begin{align*}
g(\alpha \mathbf{x}_1 + (1- \alpha)\mathbf{x}_2)
&\leq f(\alpha \mathbf{x}_1 + (1- \alpha)\mathbf{x}_2, \alpha \mathbf{y}_1 + (1- \alpha)\mathbf{y}_2)\\
&= f(\alpha (\mathbf{x}_1, \mathbf{y}_1) + (1- \alpha)(\mathbf{x}_2, \mathbf{y}_2))\\
&\leq \alpha f(\mathbf{x}_1, \mathbf{y}_1) + (1- \alpha)f(\mathbf{x}_2, \mathbf{y}_2)\\
&\leq \alpha [g(\mathbf{x}_1) + \epsilon] + (1- \alpha)[g(\mathbf{x}_2) + \epsilon]\\
&\leq \alpha g(\mathbf{x}_1) + (1- \alpha) g(\mathbf{x}_2) + \epsilon,
\end{align*}

where we used the convexity of $f$ on the second line.  Because $\epsilon > 0$ is arbitrary, the claim follows. $\square$

**EXAMPLE:** **(Distance to a Convex Set)** Let $C$ be a convex set in $\mathbb{R}^d$. We show that the distance to $C$ 

$$
g(\mathbf{x})
= \inf_{\mathbf{y} \in C} \|\mathbf{x} - \mathbf{y}\|_2,
$$

is convex. 

To apply the previous lemma, we first need to show that $f(\mathbf{x},\mathbf{y}) := \|\mathbf{x} - \mathbf{y}\|_2$ is convex as a function of $(\mathbf{x}, \mathbf{y})$. Let $\mathbf{x}_1, \mathbf{x}_2 \in \mathbb{R}^d$, $\mathbf{y}_1, \mathbf{y}_2 \in C$, and $\alpha \in [0,1]$. We want to show that $f$ evaluated at the convex combination

$$
\alpha (\mathbf{x}_1,\mathbf{y}_1)
+ (1-\alpha) (\mathbf{x}_2,\mathbf{y}_2)
= (\alpha \mathbf{x}_1 + (1-\alpha)\mathbf{x}_2, \alpha \mathbf{y}_1 + (1-\alpha)\mathbf{y}_2),
$$

is upper bounded by the same convex combination of the values of $f$ at $(\mathbf{x}_1,\mathbf{y}_1)$ and $(\mathbf{x}_2,\mathbf{y}_2)$.

By the triangle inequality and the homogeneity of the norm,

\begin{align*}
&f(\alpha \mathbf{x}_1 + (1-\alpha)\mathbf{x}_2, \alpha \mathbf{y}_1 + (1-\alpha)\mathbf{y}_2)\\
&=\|[\alpha \mathbf{x}_1 + (1-\alpha)\mathbf{x}_2] -  [\alpha \mathbf{y}_1 + (1-\alpha)\mathbf{y}_2]\|_2\\
&= \|\alpha (\mathbf{x}_1 - \mathbf{y}_1 ) + (1-\alpha)(\mathbf{x}_2 - \mathbf{y}_2)\|_2\\
&\leq \alpha\|\mathbf{x}_1 - \mathbf{y}_1\|_2 + (1-\alpha)\|\mathbf{x}_2 - \mathbf{y}_2\|_2\\
&= \alpha f(\mathbf{x}_1, \mathbf{y}_1) + (1-\alpha)f(\mathbf{x}_2, \mathbf{y}_2).
\end{align*}


It remains to show that $g(\mathbf{x}) > -\infty$ for all $\mathbf{x}$. But this is immediate since $ \|\mathbf{x} - \mathbf{y}\|_2 \geq 0$. Hence the previous lemma gives the claim. $\lhd$

**Conditions based on the gradient and Hessian** A common way to prove that a function is convex is to look at its Hessian (or second derivative in the single-variable case). But we start with a first-order characterization of convexity.

**LEMMA** **(First-Order Convexity Condition)** Let $f : \mathbb{R}^d \to \mathbb{R}$ be continuously differentiable. Then $f$ is convex if and only if 

$$
f(\mathbf{y})
\geq f(\mathbf{x}) + \nabla f(\mathbf{x})^T (\mathbf{y}-\mathbf{x}),
\qquad \forall \mathbf{x}, \mathbf{y} \in \mathbb{R}^d.
$$

$\flat$

On the right-hand side above, you should recognize the linear approximation to $f$ at $\mathbf{x}$ from *Taylor's Theorem* without the remainder. 

The condition is illustrated below.

![First-order convexity](https://i.stack.imgur.com/ehKVz.png)

([Source](https://math.stackexchange.com/questions/1518550/geometric-interpretation-of-first-order-condition))

*Proof (First-Order Convexity Condition):* Suppose first that $f(\mathbf{z}_2)
\geq f(\mathbf{z}_1) + \nabla f(\mathbf{z}_1)^T (\mathbf{z}_2-\mathbf{z}_1)$ for all $\mathbf{z}_1, \mathbf{z}_2$. For any $\mathbf{x}, \mathbf{y}$ and $\alpha \in [0,1]$, let $\mathbf{w} = (1-\alpha) \mathbf{x} + \alpha \mathbf{y}$. Then taking $\mathbf{z}_1 = \mathbf{w}$ and $\mathbf{z}_2 = \mathbf{x}$ gives

$$
f(\mathbf{x})
\geq f(\mathbf{w}) + \nabla f(\mathbf{w})^T (\mathbf{x}-\mathbf{w})
$$

and taking $\mathbf{z}_1 = \mathbf{w}$ and $\mathbf{z}_2 = \mathbf{y}$ gives

$$
f(\mathbf{y})
\geq f(\mathbf{w}) + \nabla f(\mathbf{w})^T (\mathbf{y}-\mathbf{w}).
$$

Multiplying the first inequality by $(1-\alpha)$ and the second one by $\alpha$, and adding them up gives

$$
(1-\alpha) f(\mathbf{x}) + \alpha f(\mathbf{y})
\geq f(\mathbf{w}) + \nabla f(\mathbf{w})^T ([(1-\alpha) \mathbf{x} + \alpha \mathbf{y}] - \mathbf{w})
= f(\mathbf{w})
$$

proving convexity.

For the other direction, assume that $f$ is convex. For any $\mathbf{x}, \mathbf{y}$ and $\alpha \in (0,1)$, by the *Mean Value Theorem*, for some $\xi_\alpha \in (0,1)$ it holds that

$$
f(\mathbf{w})
= f(\mathbf{x} + \alpha (\mathbf{y} - \mathbf{x}))
= f(\mathbf{x}) + \alpha (\mathbf{y} - \mathbf{x})^T \nabla f (\mathbf{x} + \xi_\alpha \alpha (\mathbf{y} - \mathbf{x}))
$$

while convexity implies

$$
f(\mathbf{w})
\leq (1-\alpha) f(\mathbf{x}) + \alpha f(\mathbf{y}).
$$

Combining, rearranging and dividing by $\alpha$ gives

$$
(\mathbf{y} - \mathbf{x})^T \nabla f (\mathbf{x} + \xi_\alpha \alpha (\mathbf{y} - \mathbf{x}))
\leq f(\mathbf{y}) - f(\mathbf{x}).
$$

Taking $\alpha \to 0$ gives the claim. $\square$

**LEMMA** **(Second-Order Convexity Condition)** Let $f : \mathbb{R}^d \to \mathbb{R}$ be twice continuously differentiable. Then $f$ is convex if and only if $H_f(\mathbf{x})$ is positive semidefinite for all $\mathbf{x} \in \mathbb{R}^d$. $\flat$

*Proof:* Suppose first that $H_f(\mathbf{z}_1) \succeq 0$ for all $\mathbf{z}_1$. For any $\mathbf{x}, \mathbf{y}$, by *Taylor*, there is $\xi \in (0,1)$ such that 

\begin{align*}
f(\mathbf{y})
&= f(\mathbf{x}) + \nabla f(\mathbf{x})^T (\mathbf{y}-\mathbf{x})
+ (\mathbf{y}-\mathbf{x})^T H_f(\mathbf{x} + \xi(\mathbf{y} - \mathbf{x})) \,(\mathbf{y}-\mathbf{x})\\
&\geq f(\mathbf{x}) + \nabla f(\mathbf{x})^T (\mathbf{y}-\mathbf{x})
\end{align*}

where we used the positive semidefiniteness of the Hessian. By the *First-Order Convexity Condition*, it implies that $f$ is convex.

For the other direction, assume that $f$ is convex. For any $\mathbf{x}, \mathbf{w}$ and $\alpha \in (0,1)$, by *Taylor* again, for some $\xi_\alpha \in (0,1)$ it holds that

\begin{align*}
f(\mathbf{x} + \alpha \mathbf{w})
= f(\mathbf{x}) + \alpha \mathbf{w}^T \nabla f (\mathbf{x})
+ \alpha^2 \mathbf{w}^T H_f(\mathbf{x} + \xi_\alpha \alpha \mathbf{w}) \,\mathbf{w}
\end{align*}

while the *First-Order Convexity Condition* implies

$$
f(\mathbf{x} + \alpha \mathbf{w})
\geq f(\mathbf{x}) + \alpha \mathbf{w}^T \nabla f (\mathbf{x}).
$$

Combining, rearranging and dividing by $\alpha^2$ gives

$$
\mathbf{w}^T H_f(\mathbf{x} + \xi_\alpha \alpha \mathbf{w}) \,\mathbf{w} \geq 0.
$$

Taking $\alpha \to 0$ and using the continuity of the Hessian shows that $\mathbf{w}^T H_f(\mathbf{x}) \,\mathbf{w} \geq 0$. Since $\mathbf{w}$ is arbitrary, this implies that the Hessian is positive semidefinite at $\mathbf{x}$. This holds for any $\mathbf{x}$, which proves the claim. $\square$

**EXAMPLE:** Consider the quadratic function 

$$
f(\mathbf{x})
= \frac{1}{2} \mathbf{x}^T P \mathbf{x} + \mathbf{q}^T \mathbf{x} + r,
$$

where $P$ is a symmetric matrix. We showed previously that the Hessian is

$$
H_f(\mathbf{x}) = \frac{1}{2}[P + P^T] = P.
$$

So $f$ is convex if and only if the matrix $P$ is positive semidefinite. $\lhd$

**EXAMPLE:** **(Log-concavity)** A function $f :\mathbb{R}^d \to \mathbb{R}$ is said to be log-concave if $-\log f$ is convex. Put differently, we require for all $\mathbf{x}, \mathbf{y} \in \mathbb{R}^d$ and $\alpha \in (0,1)$

$$
- \log f((1-\alpha)\mathbf{x} + \alpha \mathbf{y})
\leq - (1-\alpha) \log f(\mathbf{x}) - \alpha \log f(\mathbf{y}),
$$

This is equivalent to

$$
\log f((1-\alpha)\mathbf{x} + \alpha \mathbf{y}) \geq
(1-\alpha) \log f(\mathbf{x}) + \alpha \log f(\mathbf{y}) ,
$$

Or, because $a \log b = \log b^a$ and the logarithm is strictly increasing,  

$$
f((1-\alpha)\mathbf{x} + \alpha \mathbf{y}) \geq f(\mathbf{x})^{1-\alpha} f(\mathbf{y})^{\alpha}.
$$

We will see later in the course that a multivariate Gaussian vector $\mathbf{X}$ on $\mathbb{R}^d$ with mean $\bmu \in \mathbb{R}^d$ and positive definite covariance matrix $\bSigma \in \mathbb{R}^{d \times d}$ has probability density function (PDF)

$$
f_{\bmu, \bSigma}(\mathbf{x})
= \frac{1}{(2\pi)^{d/2} \,|\bSigma|^{1/2}}
\exp\left(-\frac{1}{2}(\mathbf{x} - \bmu)^T \bSigma^{-1} (\mathbf{x} - \bmu)\right)
$$

where $|A|$ is the [determinant](https://en.wikipedia.org/wiki/Determinant) of $A$, which in the case of a symmetric matrix is simply the product of its eigenvalues. We claim that this PDF is log-concave.

From the definition,

\begin{align*}
&- \log f_{\bmu, \bSigma}(\mathbf{x})\\
&= \frac{1}{2}(\mathbf{x} - \bmu)^T \bSigma^{-1} (\mathbf{x} - \bmu) + \log (2\pi)^{d/2} \,|\bSigma|^{1/2}\\
&= \frac{1}{2}\mathbf{x}^T \bSigma^{-1} \mathbf{x} 
- \bmu^T \bSigma^{-1} \mathbf{x}
+ \left[\frac{1}{2}\bmu^T \bSigma^{-1} \bmu + \log (2\pi)^{d/2} \,|\bSigma|^{1/2}\right].
\end{align*}

Let $P = \bSigma^{-1}$, $\mathbf{q} = - \bmu^T \bSigma^{-1}$ and $r = \frac{1}{2}\bmu^T \bSigma^{-1} \bmu + \log (2\pi)^{d/2} \,|\bSigma|^{1/2}$.

The previous example then implies that the PDF is log-concave if $\bSigma^{-1}$ is positive semidefinite. Since $\bSigma$ is positive definite by assumption, $\bSigma = Q \Lambda Q^T$ has a spectral decomposition where all diagonal entries of $\Lambda$ are stricly positive. Then $\bSigma^{-1} = Q \Lambda^{-1} Q^T$ where the diagonal entries of $\Lambda^{-1}$ are the inverses of those of $\Lambda$ - and hence strictly positive as well. In particular, $\bSigma^{-1}$ is positive semidefinite. $\lhd$

### Convexity and unconstrained optimization

Now comes the key property of convex functions (at least as far as we are concerned).

**Global minimization in the convex case** In the convex case, global minimization reduces to local minimization.

**THEOREM** **(Global Minimizers of Convex Functions)** Let $f : \mathbb{R}^d \to \mathbb{R}$ be a convex function. Then any local minimizer of $f$ is also a global minimizer. $\sharp$

*Proof:* By contradiction, suppose $\mathbf{x}_0$ is a local minimizer, but not a global minimizer. Then there is $\mathbf{y}$ such that 

$$
f(\mathbf{y}) < f(\mathbf{x}_0).
$$

By convexity, for any $\alpha \in (0,1)$

$$
f(\mathbf{x}_0 + \alpha (\mathbf{y} - \mathbf{x}_0))
\leq (1-\alpha) f(\mathbf{x}_0) + \alpha f(\mathbf{y}) 
< f(\mathbf{x}_0).
$$

But that implies that every open ball around $\mathbf{x}_0$ contains a point taking a smaller value than $f(\mathbf{x}_0)$, a contradiction. $\square$

In the continuously differentiable case, we get in addition that a vanishing gradient at $\mathbf{x}_0$ is now a sufficient condition for $\mathbf{x}_0$ to be a local -- and therefore global -- minimizer.

**THEOREM** **(First-Order Sufficient Condition for Convex Functions)** Let $f : \mathbb{R}^d \to \mathbb{R}$ be a continuously differentiable, convex function. If $\nabla f(\mathbf{x}_0) = \mathbf{0}$ then $\mathbf{x}_0$ is a global minimizer. $\sharp$

*Proof:* Assume $\nabla f(\mathbf{x}_0) = \mathbf{0}$. By the *First-Order Convexity Condition*, for any $\mathbf{y}$

$$
f(\mathbf{y}) - f(\mathbf{x}_0) \geq \nabla f(\mathbf{x}_0)^T (\mathbf{y} - \mathbf{x}_0) = 0.
$$

That proves the claim. $\square$

**EXAMPLE:** **(Quadratic Function)** Consider the quadratic function 

$$
f(\mathbf{x})
= \frac{1}{2} \mathbf{x}^T P \mathbf{x} + \mathbf{q}^T \mathbf{x} + r
$$

where $P$ is symmetric and positive semidefinite. The Hessian is then

$$
H_f(\mathbf{x}) = \frac{1}{2}[P + P^T] = P
$$

for any $\mathbf{x}$. So $f$ is convex. Further the gradient is

$$
\nabla f(\mathbf{x}) = P\mathbf{x} + \mathbf{q}
$$

for all $\mathbf{x}$. 

Any $\mathbf{x}$ satisfying 

$$
P\mathbf{x} = - \mathbf{q}
$$

is a global minimizer. $\lhd$

When $f$ is strictly convex, the global minimizer is unique (if it exists). (Why?)

For our purposes, we will need a uniform version of strict convexity, strong convexity, which we define in the next section.

**Strong convexity** With stronger assumptions, we obtain stronger guarantees. One such assumption is strong convexity, which we define next in the special case of twice continuously differentiable functions. 

We will use the following notation. Let $A, B \in \mathbb{R}^{d \times d}$ be symmetric matrices. Recall that $A \succeq 0$ means that $A$ is positive semidefinite. We write $A \preceq B$ (respectively $A \succeq B$) to indicate that $B - A \succeq 0$ (respectively $A - B \succeq 0$). A different, useful way to put this is the following. Recall that $B - A \succeq 0$ means $\mathbf{z}^T B\mathbf{z} - \mathbf{z}^T A\mathbf{z} \geq 0$ for all $\mathbf{z} \in \mathbb{R}^{d}$. Hence, rearranging, 

$$
A \preceq B
\iff \mathbf{z}^T A\mathbf{z} \leq \mathbf{z}^T B\mathbf{z}, \qquad \forall \mathbf{z} \in \mathbb{R}^{d}.
$$

Similarly,

$$
A \succeq B
\iff \mathbf{z}^T A\mathbf{z} \geq \mathbf{z}^T B\mathbf{z}, \qquad \forall \mathbf{z} \in \mathbb{R}^{d}.
$$

**DEFINITION** **(Strongly Convex Function)** Let $f : \mathbb{R}^d \to \mathbb{R}$ be twice continuously differentiable and let $m > 0$. We say that $f$ is $m$-strongly convex if 

$$
H_f(\mathbf{x}) \succeq m I_{d \times d},
\quad \forall \mathbf{x} \in \mathbb{R}^d.
$$ 

$\natural$

By the observation above, noting that $\mathbf{z}^T I\mathbf{z} = \|\mathbf{z}\|^2$, we get that the condition above is equivalent to

$$
\mathbf{z}^T H_f(\mathbf{x}) \,\mathbf{z} \geq m \|\mathbf{z}\|^2,
\quad \forall \mathbf{x}, \mathbf{z} \in \mathbb{R}^d.
$$

Combined with *Taylor's Theorem*, this gives immediately the following. The proof is left as an exercise (see *Exercise 4.18*). Also *Exercise 4.16* gives a practical way to check the strong convexity condition.

**LEMMA** **(Quadratic Bound for Strongly Convex Functions)** Let $f : \mathbb{R}^d \to \mathbb{R}$ be twice continuously differentiable. Then $f$ is $m$-strongly convex if and only if

$$
f(\mathbf{y})
\geq f(\mathbf{x}) 
+ \nabla f(\mathbf{x})^T(\mathbf{y} - \mathbf{x})
+ \frac{m}{2} \|\mathbf{y} - \mathbf{x}\|^2,
\qquad \forall \mathbf{x}, \mathbf{y} \in \mathbb{R}^d.
$$

$\flat$

The previous lemma immediately leads to the following fundamental result.

**THEOREM** **(Global Minimizer of Strongly Convex Functions)** Let $f : \mathbb{R}^d \to \mathbb{R}$ be twice continuously differentiable and $m$-strongly convex with $m>0$. If $\nabla f(\mathbf{x}^*) = \mathbf{0}$, then $\mathbf{x}^*$ is a unique global minimizer of $f$. $\sharp$

*Proof:* If $\nabla f(\mathbf{x}^*) = \mathbf{0}$, by the *Quadratic Bound for Strongly Convex Functions*, 

$$
f(\mathbf{y})
\geq f(\mathbf{x}^*) 
+ \frac{m}{2} \|\mathbf{y} - \mathbf{x}^*\|^2
> f(\mathbf{x}^*) 
$$

for all $\mathbf{y} \neq \mathbf{x}^*$, which proves the claim. $\square$

**EXAMPLE:** **(Quadratic Function, continued)** Consider again the quadratic function 

$$
f(\mathbf{x})
= \frac{1}{2} \mathbf{x}^T P \mathbf{x} + \mathbf{q}^T \mathbf{x} + r
$$

where $P$ is symmetric and, this time, positive definite. Again, for any $\mathbf{x}$, the Hessian is

$$
H_f(\mathbf{x}) = \frac{1}{2}[P + P^T] = P.
$$ 

So all eigenvalues of $H_f(\mathbf{x}) = P$ are stricly positive. Let $\mu > 0$ be the smallest one. Then *Exercise 4.16* implies that $f$ is $\mu$-strongly convex. 


The theorem above then indicates that there is a unique global minimizer to the least-squares objective in that case. Using previous calculation, it is obtained by computing $\mathbf{x}^* = - P^{-1} \mathbf{q}$. (Why is $P$ invertible?) $\lhd$ 

**EXAMPLE:** Consider the least-squares objective function

$$
f(\mathbf{x}) = \|A \mathbf{x} - \mathbf{b}\|^2,
$$

where $A \in \mathbb{R}^{n \times d}$ has full column rank and $\mathbf{b} \in \mathbb{R}^n$. This objective function can be rewritten as a quadratic function

\begin{align*}
f(\mathbf{x}) 
&=  \|A \mathbf{x} - \mathbf{b}\|^2\\
&= (A \mathbf{x} - \mathbf{b})^T(A \mathbf{x} - \mathbf{b})\\
&= \mathbf{x}^T A^T A \mathbf{x} - 2 \mathbf{b}^T A \mathbf{x} + \mathbf{b}^T \mathbf{b}\\
&= \frac{1}{2} \mathbf{x}^T  P \mathbf{x} + \mathbf{q}^T \mathbf{x} + r
\end{align*}

where $P = 2 A^T A$ is symmetric, $\mathbf{q} = - 2 A^T \mathbf{b}$, and $r= \mathbf{b}^T \mathbf{b} = \|\mathbf{b}\|^2$.

By the previous example, the Hessian of $f$ is

$$
H_f(\mathbf{x})
= 2 A^T A.
$$

This Hessian is positive definite. Indeed we have proved previously that, for any $\mathbf{z} \in \mathbf{R}^d$,

$$
\langle \mathbf{z}, 2 A^T A \mathbf{z}\rangle 
= 2 (A \mathbf{z})^T (A \mathbf{z})
= 2 \|A \mathbf{z}\|^2 > 0,
$$

since $A \mathbf{z} = \mathbf{0}$ implies $\mathbf{z} = \mathbf{0}$ by the full column rank assumption.

So all eigenvalues of $H_f(\mathbf{x}) = 2 A^T A$ are stricly positive. Let $\mu > 0$ be the smallest one. Then $f$ is $\mu$-strongly convex (Why?). 


The theorem above then indicates that there is a unique global minimizer to the least-squares objective in that case. $\lhd$ 

### A data science example: ridge regression

We saw in a previous subsection that the pseudoinverse gives a least norm solution to an underdetermined linear system. A similar idea turns out to be useful in the overdetermined case as well, particularly when the columns of the matrix $A$ are linearly dependent or close to linearly dependent (which is sometimes referred as [multicollinearity](https://en.wikipedia.org/wiki/Multicollinearity) in statistics). This is called ridge regression or Tikhonov regularization. It trades off minimizing the fit to the data versus minimizing the norm of the solution. More precisely, for a parameter $\lambda > 0$ to be chosen, we solve

$$
\min_{\mathbf{x} \in \mathbb{R}^m} \|A \mathbf{x} - \mathbf{b}\|_2^2 + \lambda \|\mathbf{x}\|_2^2.
$$

The second term is referred to as an $\ell_2$-regularizer. Here $A \in \mathbb{R}^{n\times m}$ with $n \geq m$ and $\mathbf{b} \in \mathbb{R}^n$.

**Solving the problem** To solve this optimization problem, we show that the objective function is strongly convex. We then find its unique stationary point. Rewriting the objective in quadratic function form

\begin{align*}
f(\mathbf{x})
&= \|A \mathbf{x} - \mathbf{b}\|_2^2 + \lambda \|\mathbf{x}\|_2^2\\
&=  \mathbf{x}^T A^T A \mathbf{x} - 2 \mathbf{b}^T A \mathbf{x} + \mathbf{b}^T \mathbf{b}
+ \lambda \mathbf{x}^T \mathbf{x}\\
&= \mathbf{x}^T (A^T A + \lambda I_{m \times m}) \mathbf{x} - 2 \mathbf{b}^T A \mathbf{x} + \mathbf{b}^T \mathbf{b}\\
&= \frac{1}{2} \mathbf{x}^T  P \mathbf{x} + \mathbf{q}^T \mathbf{x} + r,
\end{align*}

where $P = 2 (A^T A + \lambda I_{m \times m})$ is symmetric, $\mathbf{q} = - 2 A^T \mathbf{b}$, and $r= \mathbf{b}^T \mathbf{b} = \|\mathbf{b}\|^2$.

As we previously computed, the Hessian of $f$ is $H_f(\mathbf{x})= P$. Now comes a key observation. The matrix $P$ is positive definite whenever $\lambda > 0$. Indeed, for any $\mathbf{z} \in \mathbb{R}^m$,

$$
\mathbf{z}^T [2 (A^T A + \lambda I_{m \times m})] \mathbf{z}
= 2 \|A \mathbf{z}\|_2^2 + 2 \|\mathbf{z}\|_2^2 > 0.
$$

So all eigenvalues of $H_f(\mathbf{x}) = 2 (A^T A + \lambda I_{m \times m})$ are stricly positive. Let $\mu > 0$ be the smallest one. Then $f$ is $\mu$-strongly convex (Why?). This holds whether or not the columns of $A$ are linearly independent.

The stationary points are easily characterized. Recall that the gradient is $\nabla f(\mathbf{x}) = P \mathbf{x} + \mathbf{q}$. Equating to $\mathbf{0}$ leads to the system

$$
2 (A^T A + \lambda I_{m \times m}) \mathbf{x} - 2 A^T \mathbf{b} = \mathbf{0},
$$

that is

$$
\mathbf{x}^{**}
= (A^T A + \lambda I_{m \times m})^{-1} A^T \mathbf{b}.
$$

The matrix in parenthesis is invertible as it is $1/2$ of the Hessian, which is positive definite.

**Connection to SVD** Expressing the solution in terms of a compact SVD 

$$
A = U \Sigma V^T = \sum_{j=1}^r \sigma_j \mathbf{u}_j \mathbf{v}_j^T.
$$ 

provides some insights into how ridge regression works. Suppose that $A$ has full column rank. That implies that $V V^T = I_{m \times m}$. Then observe that

\begin{align*}
(A^T A + \lambda I_{m \times m})^{-1}
&= (V \Sigma U^T U \Sigma V^T + \lambda I_{m \times m})^{-1}\\
&= (V \Sigma^2 V^T + \lambda I_{m \times m})^{-1}\\
&= (V \Sigma^2 V^T + V \lambda I_{m \times m} V^T)^{-1}\\
&= (V [\Sigma^2 + \lambda I_{m \times m}] V^T)^{-1}\\
&= V (\Sigma^2 + \lambda I_{m \times m})^{-1} V^T.
\end{align*}


Hence

$$
\mathbf{x}^{**}
= (A^T A + \lambda I_{m \times m})^{-1} A^T \mathbf{b}
= V (\Sigma^2 + \lambda I_{m \times m})^{-1} V^T V \Sigma U^T \mathbf{b}
= V (\Sigma^2 + \lambda I_{m \times m})^{-1} \Sigma U^T \mathbf{b}.
$$

Our predictions are

\begin{align*}
A \mathbf{x}^{**}
&= U \Sigma V^T V (\Sigma^2 + \lambda I_{m \times m})^{-1} \Sigma U^T \mathbf{b}\\
&= U \Sigma (\Sigma^2 + \lambda I_{m \times m})^{-1} \Sigma U^T \mathbf{b}\\
&= \sum_{j=1}^r \mathbf{u}_j \left\{\frac{\sigma_j^2}{\sigma_j^2 + \lambda} \right\} \mathbf{u}_j^T \mathbf{b}.
\end{align*}

Note that the terms in curly brackets are $< 1$ when $\lambda > 0$.

Compare this to the unregularized least squares solution $\mathbf{}$, which is obtained simply by setting $\lambda = 0$ above, 

\begin{align*}
A \mathbf{x}^*
&= \sum_{j=1}^r \mathbf{u}_j  \mathbf{u}_j^T \mathbf{b}.
\end{align*}

The difference is that the regularized solution reduces (potentially significantly) the contributions from the left singular vectors corresponding to small singular values. 